# EDA 7 - Alluvials for case studies

Three types telling "counter dominated London by 2021" with three main different mechanisms.

On frame C, 452 out of 982 (~46%) MSOAs are counter-led by 2021, while only ~20% are cascade-led.

The dominant transition is inner cores flipping from cascade to counter.

Three mechanisms are:
1. Persistent cascade
   Pick cascade-led both years (actually these are only 2 areas stidfying this condition), inflow-led, modest external share.
   These are flow-defined gentrification should look like.

2. Inner cascade turning counter
   There are striking patterns that areas keeping improving on IMD but still flipping to counter flow.
   These are attribute-ascending but flow-counter areas.
   They support the out-displacement arm dominated inner London.

3. Exodus
   Harrow 008 is a 2021 cascade-extreme with inflow share dropped to 1% and a large Dom A-Dom C gap;
   Harrow 029 is a counter-to-cascade flip, at D9 with cascade-inflow-share of 0.04 and external share of 0.32.
   Both areas are the COVID affluent-outflow signature, not gentrification.
   They are flase-cascade that the national frame manufactures.

In every panel:
- inflows enter on the left, outflows exit on the right
- ribbon width is proportional to people volume
    - Red is cascade arms
        - inflow from wealthier 
        - outflow to poorer
    - Purple is counter arms
        - inflow from poorer
        - outflow to wealthier

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams['font.family'] = 'DejaVu Sans'
from pyprojroot import here

In [ ]:

ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'
OUT_DIR    = ROOT / 'outputs' / 'case_study'
NAT  = ROOT / 'outputs'/'msoa_cascade_national_frame_20260625.csv'
EDA4 = ROOT / 'outputs'/'eda4_results_for_phase3_20260626.csv'
EXT_DECILE = 6                                   # synthetic external node sits at national D6
YL = 3.05

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
C_IN, C_EX = '#c0392b', '#e8a7a0'                # cascade  internal / external (London<->outside)
K_IN, K_EX = '#6a51a3', '#bcaede'                # counter  internal / external
BAR = '#2f2f2f'

In [ ]:
def _band(ax, x0, x1, lo, hi, color):
    ax.add_patch(mpatches.Rectangle((x0, lo), x1-x0, hi-lo, facecolor=color, edgecolor='none', alpha=0.92))

In [ ]:
def _lay(segs, gap, scale):
    """segs: list of (value, color, new_arm). Lay out top-down; gap before each new arm."""
    hs = [(v/scale, c, n) for v, c, n in segs]
    tot = sum(h for h, _, _ in hs) + gap*(sum(1 for _, _, n in hs if n) - 1)
    top = tot/2; out = []
    for i, (h, c, n) in enumerate(hs):
        if n and i > 0: top -= gap
        out.append((top-h, top, c)); top -= h
    return out

In [ ]:
def arms_split(r, yr):
    """Split each national-frame arm into within-London (internal) and London<->outside (external)."""
    d  = r['Wealth_Decile_National']
    ei, eo = r[f'Ext_Inflow_nat_{yr}'], r[f'Ext_Outflow_nat_{yr}']
    iw, op = r[f'Inflow_Wealthier_nat_{yr}'], r[f'Outflow_Poorer_nat_{yr}']
    ip, ow = r[f'Inflow_Poorer_nat_{yr}'],    r[f'Outflow_Wealthier_nat_{yr}']
    e_iw = ei if d < EXT_DECILE else 0; e_ip = ei if d > EXT_DECILE else 0     # ext-in: wealthier if area<D6 else poorer
    e_ow = eo if d < EXT_DECILE else 0; e_op = eo if d > EXT_DECILE else 0     # ext-out: wealthier if area<D6 else poorer
    return dict(iw=iw, op=op, ip=ip, ow=ow, iw_e=e_iw, op_e=e_op, ip_e=e_ip, ow_e=e_ow,
                iw_i=iw-e_iw, op_i=op-e_op, ip_i=ip-e_ip, ow_i=ow-e_ow)

In [ ]:
def panel(ax, r, yr, scale, title, dom_lon):
    a = arms_split(r, yr); xL, xML, xMR, xR = 0.0, 3.3, 3.9, 7.2; gap = 0.30
    Lsegs = [(a['iw_i'],C_IN,True),(a['iw_e'],C_EX,False),(a['ip_i'],K_IN,True),(a['ip_e'],K_EX,False)]
    Rsegs = [(a['ow_i'],K_IN,True),(a['ow_e'],K_EX,False),(a['op_i'],C_IN,True),(a['op_e'],C_EX,False)]
    L, R = _lay(Lsegs, gap, scale), _lay(Rsegs, gap, scale)
    for lo, hi, c in L: _band(ax, xL, xML, lo, hi, c)
    for lo, hi, c in R: _band(ax, xMR, xR, lo, hi, c)
    bar_lo, bar_hi = min(L[-1][0], R[-1][0]), max(L[0][1], R[0][1])
    ax.add_patch(mpatches.Rectangle((xML, bar_lo), xMR-xML, bar_hi-bar_lo, facecolor=BAR, edgecolor='none'))
    iw_c=(L[0][0]+L[1][1])/2; ip_c=(L[2][0]+L[3][1])/2; ow_c=(R[0][0]+R[1][1])/2; op_c=(R[2][0]+R[3][1])/2
    def lab(x, y, tot, ext, c, ha):
        if tot/scale < 0.03: return
        ax.text(x, y, f'{int(tot)}'+(f'  (ext {int(ext)})' if ext > 0.5 else ''),
                ha=ha, va='center', fontsize=7.3, color=c)
    lab(xL-0.12, iw_c, a['iw'], a['iw_e'], C_IN, 'right'); lab(xL-0.12, ip_c, a['ip'], a['ip_e'], K_IN, 'right')
    lab(xR+0.12, ow_c, a['ow'], a['ow_e'], K_IN, 'left');  lab(xR+0.12, op_c, a['op'], a['op_e'], C_IN, 'left')
    casc, cnt = a['iw']+a['op'], a['ip']+a['ow']; dom = casc/(casc+cnt)
    infl = a['iw']/(a['iw']+a['op']) if (a['iw']+a['op']) else np.nan
    ext_sh = (a['iw_e']+a['ip_e']+a['ow_e']+a['op_e'])/(casc+cnt) if (casc+cnt) else 0
    ax.text(0.5, -0.03, f'dominance(nat) {dom:.2f}   inflow share {infl:.2f}   external {ext_sh:.0%}'
            f'   ·   dominance(London) {dom_lon:.2f}', transform=ax.transAxes, ha='center', va='top',
            fontsize=8, color=C_IN if dom >= 0.5 else K_IN, fontweight='bold')
    ax.text(0.5, 1.0, title, transform=ax.transAxes, ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xlim(-1.75, 8.95); ax.set_ylim(-YL, YL); ax.set_axis_off()
 


In [ ]:
nat = pd.read_csv(NAT); e4 = pd.read_csv(EDA4)
domA = lambda c, yr: e4.loc[e4['msoa11cd']==c, f'Dom_A_{yr}'].iloc[0]
row  = lambda c: nat[nat['msoa11cd']==c].iloc[0]

In [ ]:
def figscale(cases):
    m = 0
    for c, _ in cases:
        r = row(c)
        for yr in ['11','21']:
            a = arms_split(r, yr); m = max(m, a['iw']+a['ip'], a['ow']+a['op'])
    return m/3.0

In [ ]:
LEG = [mpatches.Patch(color=C_IN, label='Cascade · within London'),
       mpatches.Patch(color=C_EX, label='Cascade · London - rest of England'),
       mpatches.Patch(color=K_IN, label='Counter · within London'),
       mpatches.Patch(color=K_EX, label='Counter · London - rest of England')]

In [ ]:
def build(title, cases, fname, caption):
    sc = figscale(cases); nr = len(cases)
    fig, axes = plt.subplots(nr, 2, figsize=(13, 3.0*nr+1.6), squeeze=False)
    for i, (c, n) in enumerate(cases):
        panel(axes[i,0], row(c), '11', sc, f'{n}  ·  2011', domA(c,'11'))
        panel(axes[i,1], row(c), '21', sc, f'{n}  ·  2021', domA(c,'21'))
    fig.suptitle(title, fontsize=13.5, fontweight='bold', y=0.995)
    fig.legend(handles=LEG, loc='lower center', ncol=2, frameon=False, fontsize=8.4, bbox_to_anchor=(0.5,0.004))
    fig.text(0.5, 0.055, 'Inflows enter on the left, outflows exit on the right.  ' + caption,
             ha='center', va='bottom', fontsize=8.2, style='italic', color='#333', wrap=True)
    plt.tight_layout(rect=[0,0.085,1,0.97]); plt.savefig(OUT_DIR / fname, dpi=130, bbox_inches='tight'); plt.show()

build('Persistent cascade — affluent inflow AND poorer outflow, robust across frames',
      [('E02000191','Camden 026'),('E02000873','Tower Hamlets 010')],
      'fig_case_A_persistent_cascade.png',
      'Affluent-IN (red, left) and poorer-OUT (red, right) are both thick and the national/London dominance gap is small. '
      'Light-red on the inflow arm shows most affluent in-migration arrives from outside London —-> a real gentrification cascade.')
 
build('Inner cascade turning counter — the city-wide 2021 reversal',
      [('E02000809','Southwark 003'),('E02000561','Islington 008'),('E02000957','Wandsworth 035')],
      'fig_case_B_cascade_to_counter.png',
      'These inner cores keep improving on IMD yet the FLOW balance tips from red (cascade) to purple (counter) by 2021: '
      'the affluent-OUT / poorer-IN arms grow relative to the cascade arms (dominance crosses 0.50; the figure colour flips).')
 
build('Exodus — outer high-decile, COVID-era affluent outflow',
      [('E02000440','Harrow 008'),('E02000461','Harrow 029')],
      'fig_case_C_exodus.png',
      'The affluent-inflow arm is a bare thread (inflow share ~0.01-0.04). The cascade is almost all poorer-OUTflow, much of it '
      'light-red London->outside moves. National frame reads cascade, London frame reads neutral/counter — the gap is the exodus signature.')